# Aula 2.2 — Feature Engineering com Databricks Feature Store 🚀

Nesta aula, vamos abordar um dos maiores desafios ao colocar modelos de "Machine Learning" em produção: a consistência entre o momento do treino e o momento da inferência, além de mitigar o "data leakage" (vazamento de dados).

## 1. O Problema de Consistência Treino-Inferência
Em sistemas tradicionais, o cientista de dados cria variáveis complexas durante o treinamento. No entanto, o time de engenharia precisa reescrever toda essa lógica para rodar em tempo real na API de produção. Esse descasamento gera erros graves e perda de performance do modelo ("Feature Drift").

A **Feature Store** do Databricks, governada pelo **Unity Catalog**, resolve isso atuando como um repositório centralizado de recursos:
1. **Fonte Única da Verdade:** O mesmo cálculo usado no treino é disponibilizado em baixa latência para a produção.
2. **"Point-in-time correctness":** Evita o "data leakage" usando o "time travel" do Delta Lake para garantir que o modelo só olhe para o passado exato do evento durante o treino.
3. **Qualidade de Dados:** Tabelas de recursos centralizadas garantem padronização de tipos e linhagem de dados nativa.

---

## Dataset Iris
Para focar puramente na sintaxe e arquitetura do `FeatureEngineeringClient`, utilizaremos o clássico "dataset" **Iris**. Como as "Feature Tables" do Unity Catalog exigem uma **Chave Primária**, adicionaremos programaticamente uma coluna de identificação única (`id_flor`).

In [0]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

iris estava no volume inbound, o código abaixo é pra mover pra workspace default

In [0]:
%sql
-- Move a tabela iris_dataset do catálogo inbound.default para workspace.default
-- Isso garante que o Cell 4 consiga ler spark.table("workspace.default.iris_dataset")
-- ALTER TABLE RENAME não suporta mudança de catálogo, então usamos CTAS + DROP

-- Se já existir uma tabela com esse nome no destino, remova antes de copiar

--DROP TABLE IF EXISTS workspace.default.iris_dataset;

-- Cria a tabela de destino copiando os dados da origem (catálogo inbound)

--CREATE TABLE workspace.default.iris_dataset AS SELECT * FROM inbound.default.iris_dataset;

-- Remove a tabela original do catálogo inbound

--DROP TABLE inbound.default.iris_dataset;

-- Verifica o resultado

--SHOW TABLES IN workspace.default LIKE 'iris_dataset';

In [0]:
import pandas as pd
from sklearn.datasets import load_iris
from pyspark.sql.functions import col, expr, rand

# Catalogo e Esquema do dataset Iris
catalogo_origem = "workspace"
esquema_origem = "default"
tabela_iris_raw = f"{catalogo_origem}.{esquema_origem}.iris_dataset"

# Lendo os dados brutos que já estão no ambiente
df_iris = spark.table(tabela_iris_raw)

# Garanta que a tabela possua a chave primária (id_flor) e as colunas ajustadas.

# Registra o DataFrame atual como uma View temporária no Spark
df_iris.createOrReplaceTempView("vw_iris_raw")

# Executa a query SQL para construir a chave primária
df_iris_sql = spark.sql("""
    SELECT 
        CONCAT('flor_', ROW_NUMBER() OVER (ORDER BY (SELECT NULL))) AS id_flor,
        sepal_length,
        sepal_width,
        petal_length,
        petal_width,
        species
    FROM vw_iris_raw
""")

# Sobrescreve o DataFrame original com o resultado do SQL
df_iris_new = df_iris_sql

display(df_iris_new.limit(5))


In [0]:
from databricks import feature_engineering

# Definindo o destino das nossas features
nome_tabela_features = f"{catalogo_origem}.{esquema_origem}.iris_features_table"

# Instanciando o cliente de engenharia Databricks
fe = feature_engineering.FeatureEngineeringClient()

# Isolando apenas os recursos (Features) e a chave primária que vão para a store
# Removemos a coluna target (species) para que ela não seja exposta na tabela de recursos
df_features_only = df_iris_new.select("id_flor", "sepal_length", "sepal_width", "petal_length", "petal_width")

print(f" Gravando e registrando a Feature Table em: {nome_tabela_features}...")

# Criando e populando a tabela de recursos de forma centralizada
fe.create_table(
    name=nome_tabela_features,
    primary_keys=["id_flor"],
    df=df_features_only,
    schema=df_features_only.schema,
    description="Tabela de recursos morfológicos das flores Iris integrada à Feature Store do Unity Catalog."
)
print("Features salvas e sincronizadas com sucesso! Você já consegue visualizá-las pelo Catalog Explorer.")

In [0]:
from pyspark.sql.functions import col

# 1. Carrega a tabela de recursos atual a partir da Feature Store para derivar a nova informação
df_existente = spark.table(nome_tabela_features)

print("Calculando a nova feature: razão morfológica da pétala (petal_ratio)...")

# Criando a nova variável baseada em colunas existentes (Razão = Comprimento / Largura)
# Mantemos obrigatoriamente a chave primária 'id_flor' no DataFrame resultante
df_nova_feature = df_existente.select(
    col("id_flor"),
    (col("petal_length") / col("petal_width")).alias("petal_ratio")
)

print(f"Atualizando a Feature Table no Unity Catalog: {nome_tabela_features}...")

# Gravando a nova característica usando o write_table
# O Databricks faz o merge inteligente baseado na chave primária ['id_flor']
fe.write_table(
    name=nome_tabela_features,
    df=df_nova_feature,
    mode="merge" # Realiza o update das linhas combinando as chaves primárias
)

print("Nova feature 'petal_ratio' adicionada e integrada com sucesso!")

# Visualizando o novo esquema atualizado
display(spark.table(nome_tabela_features).limit(5))

In [0]:
%sql
ALTER TABLE workspace.default.iris_features_table SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.minReaderVersion' = '2', 'delta.minWriterVersion' = '5');
ALTER TABLE workspace.default.iris_features_table DROP COLUMNS (petal_ratio);

In [0]:
%sql
SELECT * FROM workspace.default.iris_features_table

In [0]:
%sql
DESCRIBE HISTORY workspace.default.iris_features_table

### Qual a diferença entre uma Tabela Delta Comum e uma "Feature Table"?

Para quem olha de fora (via comandos `SELECT`), ambas parecem exatamente iguais. No entanto, ao registrar seus dados usando o `FeatureEngineeringClient` em vez de um simples `.write.saveAsTable()`

A tabela abaixo resume as diferenças fundamentais:

| Funcionalidade / Recurso | Tabela Delta Comum  | Feature Store |
| :--- | :--- | :--- |
| **Formato Físico** | Delta Lake (Arquivos Parquet) | Delta Lake (Arquivos Parquet) |
| **Chave Primária Obrigatória** | Não exigida. | **Obrigatória**. Define o índice de busca (ex: `id_flor`, `id_usuario`). |
| **Rastreabilidade de IA (Linhagem)** | Sabe apenas quais tabelas geraram os dados. | Sabe quais modelos, "notebooks" e "pipelines" consomem cada coluna. |
| **Prevenção de Vazamento de Dados** | Manual. Você precisa codificar os `JOINs` temporais na unha. | **Automatizada**. O "Point-in-time correctness" usa o "timestamp" para evitar "data leakage". |
| **Integração com APIs de Produção** | Exige engenharia para reescrever consultas SQL em tempo real. | **Nativa**. O "Model Serving" busca os dados sozinho em "Online Tables" de baixa latência. |

---

### O Cenário Prático na Empresa:

1. **Se você salva como Tabela Comum:** O cientista de dados cria a feature `media_compras_30d`. Se outro cientista precisar usar essa mesma variável em outro modelo, ele terá que adivinhar qual é o script de engenharia de dados, recalculá-la por conta própria (correndo o risco de usar regras diferentes) e gerar duplicidade de dados no "lake".
2. **Se você salva na Feature Store:** O recurso fica indexado no painel do **Catalog Explorer**. Qualquer pessoa da empresa pode pesquisar por `media_compras_30d`, entender quem a criou, ver o gráfico de linhagem e consumi-la instantaneamente com uma linha de código, garantindo governança absoluta e reuso de código.